In [ ]:
# Install muldataframe in colab.!pip install muldataframe

In [ ]:
import pandas as pd
import muldataframe as md

### Initialization

In [4]:
index = pd.DataFrame([[1,2],[3,6],[5,6]],
                     index=['a','b','b'],
                     columns=['x','y'])
columns = pd.DataFrame([[5,7],[3,6]],
                    index=['c','d'],
                    columns=['f','g'])
mf = md.MulDataFrame([[1,2],[8,9],[8,7]],
    index=index,columns=columns)

In [9]:
mf

(3, 2)    g  7  6
          f  5  3
             c  d
--------  ---------
   x  y      c  d
a  1  2   a  1  2
b  3  6   b  8  9
b  5  6   b  8  7

### Primary indexing

In [5]:
mf.loc['b']

(2, 2)    g  7  6
          f  5  3
             c  d
--------  ---------
   x  y      c  d
b  3  6   b  8  9
b  5  6   b  8  7

In [6]:
mf['d']

(3,)     g  6
         f  3
            d
-------  ------
   x  y     d
a  1  2  a  2
b  3  6  b  9
b  5  6  b  7

In [7]:
mf.iloc[[0,1],[0]]

(2, 1)    g  7
          f  5
             c
--------  ------
   x  y      c
a  1  2   a  1
b  3  6   b  8

### Successive indexing

In [8]:
mf.mloc[[..., 6],[3]]

(2,)     g  6
         f  3
            d
-------  ------
   x  y     d
b  3  6  b  9
b  5  6  b  7

In [11]:
mf.mloc[{'y':[2,6],'x':[3]}]

(1, 2)    g  7  6
          f  5  3
             c  d
--------  ---------
   x  y      c  d
b  3  6   b  8  9

### Pandas MultiIndex behavior comparison

In [17]:
df = mf.df
df.index = pd.MultiIndex.from_frame(mf.index)
df.columns = pd.MultiIndex.from_frame(mf.columns)
ix = pd.IndexSlice
df

,f,5,3
,g,7,6
x,y,,
1,2,1,2
3,6,8,9
5,6,8,7


In [21]:
print(type(df.loc[ix[:,6],ix[3]]))
df.loc[ix[:,6],ix[3]]

<class 'pandas.core.frame.DataFrame'>


,g,6
x,y,
3,6,9
5,6,7


In [22]:
print(type(df.loc[ix[:,6],ix[3,6]]))
df.loc[ix[:,6],ix[3,6]]

<class 'pandas.core.series.Series'>


x  y
3  6    9
5  6    7
Name: (3, 6), dtype: int64

In [20]:
df.loc[ix[:,6]]

KeyError: 6

### Metadata-aware query examples


In [23]:
mf.query('d < 9',index='y==6')

(1, 2)    g  7  6
          f  5  3
             c  d
--------  ---------
   x  y      c  d
b  5  6   b  8  7

In [24]:
mf.query('d < 9',index='y==6',columns='f==5')

(1, 1)    g  7
          f  5
             c
--------  ------
   x  y      c
b  5  6   b  8

### Function call example

In [25]:
mf.call("sum", axis=1) 

(3,)     Series([], Name: sum, dtype: object)
-------  --------------------------------------
   x  y     sum
a  1  2  a    3
b  3  6  b   17
b  5  6  b   15

### Groupby iteration example

In [26]:
for key, group in mf.groupby('y'):
    if key == 6:
        print(key,'\n',group)


6 
 (2, 2)    g  7  6
          f  5  3
             c  d
--------  ---------
   x  y      c  d
b  3  6   b  8  9
b  5  6   b  8  7


### Groupby aggregation example

In [27]:
mf.groupby('y').mean(axis=0)

(2, 2)           g  7  6
                 f  5  3
                    c  d
---------------  -----------
Empty DataFrame       c    d
Columns: []      y
Index: [2, 6]    2  1.0  2.0
                 6  8.0  8.0

In [28]:
mf.groupby('y',agg_mode='list',keep_primary=True).mean()

(2, 2)                   g  7  6
                         f  5  3
                            c  d
-----------------------  -----------
  primary_index       x       c    d
y                        y
2             a       1  2  1.0  2.0
6             b  [3, 5]  6  8.0  8.0